In [5]:
import numpy as np
import tensorflow as tf

from data_loader import EllipticDatasetLoader
from models import EvolveGCN
from models.layers import EGCUH, HGRUCell, SummarizeLayer, GCNLayer

In [9]:
DATADIR = "/home/karim/Projects/Crypto4GraphAI/AML_EvolveGCN/EvolveGCN/data/elliptic_bitcoin_dataset/"
FILTER_UNKNOWN = False
ONLY_LOCAL_FEATURE = False
CLASS_WEIGTHS = [0.7,0.29,0.01]
NUM_ROLLS = 4
TEST_SHARE = 0.3
NUM_EPOCH = 20
LEARNING_RATE = 1e-3

In [7]:
def reset_metrics(list_of_metrics):
    for m in list_of_metrics:
        m.reset_states()

In [8]:
dl = EllipticDatasetLoader(DATADIR, TEST_SHARE, FILTER_UNKNOWN, local_features_only = ONLY_LOCAL_FEATURE)

model = EvolveGCN([
    EGCUH(HGRUCell(64),SummarizeLayer(),activation="relu"),
    EGCUH(HGRUCell(dl.num_classes),SummarizeLayer())
])

optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
loss_func = tf.keras.losses.CategoricalCrossentropy(from_logits=True)

In [10]:
def run_model(adj,nodes,targets,training=False):
    weigths = tf.reduce_sum(CLASS_WEIGTHS * targets, axis=-1)
    states = model.get_initial_weigths(tf.shape(nodes))
    logits = []
    for i in range(NUM_ROLLS):
        l, states = model([adj, nodes, states], training=training)
        logits.append(l)

    loss = sum(loss_func(targets, l, sample_weight=weigths) for l in logits)
    return logits[-1], loss, weigths

In [11]:
train_loss_metric = tf.keras.metrics.Mean()
train_accuracy_metric = tf.keras.metrics.Accuracy()
train_precision_metric = tf.keras.metrics.Precision()
train_recall_metric = tf.keras.metrics.Recall()
test_loss_metric = tf.keras.metrics.Mean()
test_accuracy_metric = tf.keras.metrics.Accuracy()
test_precision_metric = tf.keras.metrics.Precision()
test_recall_metric = tf.keras.metrics.Recall()
metrics = [train_loss_metric,train_accuracy_metric,train_precision_metric,train_recall_metric
            ,test_loss_metric,test_accuracy_metric,test_precision_metric,test_recall_metric]

In [12]:
for epoch in range(NUM_EPOCH):
    reset_metrics(metrics)

    for _, n, t, adj in dl.train_batch_iterator():
        with tf.GradientTape() as tape:
            logits, loss, weigths = run_model(adj, n, t, training=True)

        grads = tape.gradient(loss, model.trainable_weights)
        optimizer.apply_gradients(zip(grads,model.trainable_weights))

        y_true = tf.cast(tf.argmax(t,axis=-1) == 0,tf.float32)
        y_pred = tf.cast(tf.argmax(logits,axis=-1) == 0,tf.float32)
        train_loss_metric(loss)
        train_accuracy_metric(tf.argmax(t,axis=-1), tf.argmax(logits,axis=-1), sample_weight=weigths)
        train_precision_metric(y_true, y_pred)
        train_recall_metric(y_true, y_pred)

    for _, n, t, adj in dl.test_batch_iterator():
        logits, loss, weigths = run_model(adj, n, t)

        y_true = tf.cast(tf.argmax(t, axis=-1) == 0, tf.float32)
        y_pred = tf.cast(tf.argmax(logits, axis=-1) == 0, tf.float32)
        test_loss_metric(loss)
        test_accuracy_metric(tf.argmax(t,axis=-1), tf.argmax(logits,axis=-1), sample_weight=weigths)
        test_precision_metric(y_true, y_pred)
        test_recall_metric(y_true, y_pred)

    print("Epoch: {}\nTRAIN Loss: {:.5}| Accuracy: {:.4}| Precision: {:.4}| Recall: {:.4}\nTEST Loss: {:.4}| Accuracy: {:.4}| Precision: {:.4}| Recall: {:.4}".format(
        epoch, train_loss_metric.result().numpy(), train_accuracy_metric.result().numpy(),
        train_precision_metric.result().numpy(),train_recall_metric.result().numpy(),
        test_loss_metric.result().numpy(), test_accuracy_metric.result().numpy(),
        test_precision_metric.result().numpy(),test_recall_metric.result().numpy()
    ))

Epoch: 0
TRAIN Loss: 3.9088e+04| Accuracy: 0.1422| Precision: 0.03369| Recall: 0.3033
TEST Loss: 2.65e+03| Accuracy: 0.1253| Precision: 0.03058| Recall: 0.4321
Epoch: 1
TRAIN Loss: 2.0335e+04| Accuracy: 0.1535| Precision: 0.04061| Recall: 0.329
TEST Loss: 1.74e+03| Accuracy: 0.1319| Precision: 0.03419| Recall: 0.4718
Epoch: 2
TRAIN Loss: 4.2789e+04| Accuracy: 0.2101| Precision: 0.03638| Recall: 0.3359
TEST Loss: 1.473e+03| Accuracy: 0.1376| Precision: 0.02803| Recall: 0.458
Epoch: 3
TRAIN Loss: 2.9879e+04| Accuracy: 0.2766| Precision: 0.03489| Recall: 0.4287
TEST Loss: 1.814e+03| Accuracy: 0.1849| Precision: 0.02582| Recall: 0.4275
Epoch: 4
TRAIN Loss: 2101.1| Accuracy: 0.2784| Precision: 0.03175| Recall: 0.3411
TEST Loss: 1.49e+03| Accuracy: 0.1429| Precision: 0.03046| Recall: 0.5383
Epoch: 5
TRAIN Loss: 4.632e+04| Accuracy: 0.3137| Precision: 0.03543| Recall: 0.4252
TEST Loss: 1.477e+03| Accuracy: 0.1381| Precision: 0.02011| Recall: 0.4645
Epoch: 6
TRAIN Loss: 2.1856e+04| Accuracy: 0